# Historical COVID-19 Next-Day Case Modeling

**Recruiter-facing end-to-end analysis · Historical panel forecasting · Python 3.12/3.13**

> Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.

## Executive summary

**Objective:** Compare lag-only one-day-ahead models with transparent baselines across multiple chronological windows.

**Data:** 19,496 country-date rows from December 2019 through May 2020 with cases, tests, policy, population, and demographic fields.

**Verified result:** Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.

**Decision supported:** Assess baseline adequacy and reporting risk—not make a current forecast or policy recommendation.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A public-health data scientist reviewing historical methods.

**Decision:** Assess baseline adequacy and reporting risk—not make a current forecast or policy recommendation.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '13-covid-outbreak-prediction'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 13-covid-outbreak-prediction


## 4. Data provenance and scope

Schema and period align with an early Our World in Data snapshot; exact snapshot checksum and redistribution history were not recorded in the original repository.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

     file  size_mb           sha256
covid.csv    2.999 04999b024586aafb


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


covid.csv: 32 columns
iso_code location       date  total_cases  new_cases  total_deaths  ...  cvd_death_rate  diabetes_prevalence  female_smokers  male_smokers  handwashing_facilities  hospital_beds_per_100k
     ABW    Aruba 2020-03-13            2          2             0  ...             NaN                11.62             NaN           NaN                     NaN                     NaN
     ABW    Aruba 2020-03-20            4          2             0  ...             NaN                11.62             NaN           NaN                     NaN                     NaN
     ABW    Aruba 2020-03-24           12          8             0  ...             NaN                11.62             NaN           NaN                     NaN                     NaN
     ABW    Aruba 2020-03-25           17          5             0  ...             NaN                11.62             NaN           NaN                     NaN                     NaN
     ABW    Aruba 2020-03-26           19 

## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 146 lines
Functions: _ridge_pipeline, _boosting_pipeline, run_analysis


## 7. Methodology and hypotheses

Revision audit, lag/rolling features, three 14-day rolling origins, last-value and rolling baselines, country-aware ridge, histogram boosting, final chronological holdout, diagnostic intervals, and country-level error analysis.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_13_covid_outbreak_prediction", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 2.34 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (32 fields)
                  column   dtype  missing_count  missing_percent  unique_values  constant
                iso_code  object             64            0.328            212     False
                location  object              0            0.000            212     False
                    date  object              0            0.000            146     False
             total_cases   int64              0            0.000           5210     False
               new_cases   int64              0            0.000           1757     False
            total_deaths   int64              0            0.000           1780     False
              new_deaths   int64              0            0.000            562     False
 total_cases_per_million float64            377            1.934          10620     False
   new_cases_per_million float64            377            1.934           6169     False
total_deaths_per_million float64            377            1.934      

## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'model_comparison.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.')

Primary evidence: model_comparison.csv, shape=(4, 5)
                      model    mean_rmse     std_rmse     mean_mae  mean_r_squared
         rolling_7_baseline 4.998500e+02 8.722310e+01 8.988680e+01    9.465000e-01
                 last_value 5.708421e+02 1.775552e+02 8.821420e+01    9.259000e-01
hist_gradient_boosting_lags 9.354031e+02 4.970915e+02 1.250463e+02    7.875000e-01
         ridge_country_lags 2.217757e+13 3.841268e+13 6.633544e+11   -2.934194e+20

Verified result:
Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.


## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "selection": "three rolling-origin 14-day windows",
  "final_holdout_start": "2020-05-11",
  "final_holdout_end": "2020-05-24",
  "final_train_rows": 14890,
  "final_test_rows": 2917,
  "random_seed": 42
}

DIAGNOSTIC_90_PERCENT_INTERVAL
{
  "absolute_error_radius_from_validation": 142.28571428571433,
  "final_test_coverage": 0.8961261570106274
}


## 12. Visual evidence

### Chronological Holdout

![chronological_holdout](../reports/figures/chronological_holdout.png)

### Historical Covid Validation Evidence

![historical_covid_validation_evidence](../reports/figures/historical_covid_validation_evidence.png)

## 13. Business interpretation

Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.

The correct action is to use this result as evidence for **Assess baseline adequacy and reporting risk—not make a current forecast or policy recommendation.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Historical educational analysis only. Early-pandemic reporting revisions, testing changes, policy shifts, and short coverage make it unsuitable for current forecasting, policy, or medical decisions.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                                artifact  size_kb       sha256
               reports/figures/chronological_holdout.png    108.1 519ab733241d
reports/figures/historical_covid_validation_evidence.png    265.4 604ce770638c
                                    reports/metrics.json      3.7 cd03fde5bf93
             reports/tables/aggregated_test_forecast.csv      1.0 08e422e702d7
                         reports/tables/data_quality.csv      1.5 66c504bd92ca
                     reports/tables/model_comparison.csv      0.4 52050bff267a
               reports/tables/rolling_origin_results.csv      1.5 749c30b35eec
                reports/tables/test_error_by_country.csv      7.1 5a90bdb38627


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed compare lag-only one-day-ahead models with transparent baselines across multiple chronological windows. using revision audit, lag/rolling features, three 14-day rolling origins, last-value and rolling baselines, country-aware ridge, histogram boosting, final chronological holdout, diagnostic intervals, and country-level error analysis. The final verified conclusion is: **Across rolling origins, the seven-day rolling baseline remains best; final historical holdout RMSE is 388.2 with 89.6% diagnostic interval coverage.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/13-covid-outbreak-prediction/src/analysis.py
python scripts/execute_notebooks.py --project 13-covid-outbreak-prediction
```